# Benchmark: Folding Severity Sweep

Generates DVFs with increasing levels of folding by scaling displacement
magnitude, and measures how the optimizer copes:

- **Magnitude sweep**: scale a base random field from mild to extreme
- **Density sweep**: increase the number of folding sources (correspondences)

Metrics: initial neg-Jdet count, runtime, final L2 error, convergence success.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

from dvfopt import (
    iterative_parallel,
    jacobian_det2D,
    generate_random_dvf,
    scale_dvf,
)
import dvfopt.laplacian as laplacian

---
## Part 1: Magnitude Sweep

A fixed 5x5 random pattern is upscaled to 40x40 with varying
`max_magnitude` values.  Higher magnitude → more aggressive
displacements → more folding.

In [ ]:
MAGNITUDES = [1.0, 2.0, 3.0, 5.0, 7.0, 10.0, 15.0, 20.0]
TARGET_SIZE = (40, 40)
SEED = 42

In [ ]:
mag_results = {}

print(f"{'Magnitude':>10s}  {'Neg init':>10s}  {'Neg final':>10s}  "
      f"{'Time (s)':>10s}  {'Min Jdet':>10s}  {'L2 Error':>10s}")
print("-" * 72)

for mag in MAGNITUDES:
    dvf = generate_random_dvf((3, 1, 5, 5), mag, SEED)
    dvf = scale_dvf(dvf, TARGET_SIZE)

    phi_init = np.stack([dvf[-2, 0], dvf[-1, 0]])
    jac_init = jacobian_det2D(phi_init)
    n_neg_init = int((jac_init <= 0).sum())

    t0 = time.perf_counter()
    phi = iterative_parallel(dvf.copy(), verbose=0)
    elapsed = time.perf_counter() - t0

    jac_final = jacobian_det2D(phi)
    n_neg_final = int((jac_final <= 0).sum())
    min_jdet = float(jac_final.min())
    l2 = float(np.sqrt(np.sum((phi - phi_init) ** 2)))

    mag_results[mag] = {
        "n_neg_init": n_neg_init,
        "n_neg_final": n_neg_final,
        "time": elapsed,
        "min_jdet": min_jdet,
        "l2_err": l2,
    }

    status = "OK" if n_neg_final == 0 else "FAIL"
    print(f"{mag:>10.1f}  {n_neg_init:>10d}  {n_neg_final:>10d}  "
          f"{elapsed:>10.2f}  {min_jdet:>+10.6f}  {l2:>10.4f}  [{status}]")

In [ ]:
mags = list(mag_results.keys())
m_negs = [mag_results[m]["n_neg_init"] for m in mags]
m_times = [mag_results[m]["time"] for m in mags]
m_l2s = [mag_results[m]["l2_err"] for m in mags]
m_success = [mag_results[m]["n_neg_final"] == 0 for m in mags]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

ax = axes[0]
ax.plot(mags, m_negs, "o-", color="tab:red")
ax.set_xlabel("Max displacement magnitude")
ax.set_ylabel("Neg Jdet pixels (initial)")
ax.set_title("Folding vs Magnitude")
ax.grid(True, alpha=0.3)

ax = axes[1]
colors = ["tab:green" if s else "tab:red" for s in m_success]
ax.bar([str(m) for m in mags], m_times, color=colors)
ax.set_xlabel("Max magnitude")
ax.set_ylabel("Time (s)")
ax.set_title("Runtime (green=converged, red=failed)")

ax = axes[2]
ax.plot(m_negs, m_l2s, "s-", color="tab:orange")
ax.set_xlabel("Initial neg-Jdet pixels")
ax.set_ylabel("L2 Error")
ax.set_title("L2 Cost vs Folding Amount")
ax.grid(True, alpha=0.3)

plt.suptitle("Magnitude Sweep — 40x40 grid", fontsize=13)
plt.tight_layout()
plt.show()

---
## Part 2: Folding Density Sweep

Varies the number of crossing correspondence pairs on a fixed 30x30 grid.
More pairs → more spatially distributed folding regions.

In [ ]:
def make_crossing_pairs(n_pairs, H, W, seed=0):
    """Generate n crossing correspondence pairs spread across the grid."""
    rng = np.random.default_rng(seed)
    margin = 3
    msample = []
    fsample = []

    for _ in range(n_pairs):
        # Two nearby points that swap positions
        y1 = rng.integers(margin, H - margin)
        x1 = rng.integers(margin, W - margin)
        dy = rng.integers(2, 5)
        dx = rng.integers(2, 5)
        y2 = min(y1 + dy, H - margin - 1)
        x2 = min(x1 + dx, W - margin - 1)

        msample.append([0, y1, x1])
        msample.append([0, y2, x2])
        fsample.append([0, y2, x2])
        fsample.append([0, y1, x1])

    return np.array(msample), np.array(fsample)

In [ ]:
PAIR_COUNTS = [1, 2, 4, 6, 8, 12, 16, 20]
GRID_H, GRID_W = 30, 30

density_results = {}

print(f"{'Pairs':>8s}  {'Neg init':>10s}  {'Neg final':>10s}  "
      f"{'Time (s)':>10s}  {'Min Jdet':>10s}  {'L2 Error':>10s}")
print("-" * 68)

for n_pairs in PAIR_COUNTS:
    ms, fs = make_crossing_pairs(n_pairs, GRID_H, GRID_W, seed=n_pairs)
    fixed_sample = np.zeros((1, GRID_H, GRID_W))
    dvf, *_ = laplacian.sliceToSlice3DLaplacian(fixed_sample, ms, fs)

    phi_init = np.stack([dvf[-2, 0], dvf[-1, 0]])
    jac_init = jacobian_det2D(phi_init)
    n_neg_init = int((jac_init <= 0).sum())

    t0 = time.perf_counter()
    phi = iterative_parallel(dvf.copy(), verbose=0)
    elapsed = time.perf_counter() - t0

    jac_final = jacobian_det2D(phi)
    n_neg_final = int((jac_final <= 0).sum())
    min_jdet = float(jac_final.min())
    l2 = float(np.sqrt(np.sum((phi - phi_init) ** 2)))

    density_results[n_pairs] = {
        "n_neg_init": n_neg_init,
        "n_neg_final": n_neg_final,
        "time": elapsed,
        "min_jdet": min_jdet,
        "l2_err": l2,
    }

    status = "OK" if n_neg_final == 0 else "FAIL"
    print(f"{n_pairs:>8d}  {n_neg_init:>10d}  {n_neg_final:>10d}  "
          f"{elapsed:>10.2f}  {min_jdet:>+10.6f}  {l2:>10.4f}  [{status}]")

In [ ]:
pairs = list(density_results.keys())
d_negs = [density_results[p]["n_neg_init"] for p in pairs]
d_times = [density_results[p]["time"] for p in pairs]
d_l2s = [density_results[p]["l2_err"] for p in pairs]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

ax = axes[0]
ax.plot(pairs, d_negs, "o-", color="tab:red")
ax.set_xlabel("Number of crossing pairs")
ax.set_ylabel("Neg Jdet pixels (initial)")
ax.set_title("Folding vs Pair Count")
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(pairs, d_times, "o-", color="tab:blue")
ax.set_xlabel("Number of crossing pairs")
ax.set_ylabel("Time (s)")
ax.set_title("Runtime vs Pair Count")
ax.grid(True, alpha=0.3)

ax = axes[2]
ax.plot(pairs, d_l2s, "s-", color="tab:orange")
ax.set_xlabel("Number of crossing pairs")
ax.set_ylabel("L2 Error")
ax.set_title("L2 Cost vs Pair Count")
ax.grid(True, alpha=0.3)

plt.suptitle(f"Density Sweep — {GRID_H}x{GRID_W} grid", fontsize=13)
plt.tight_layout()
plt.show()

---
## Combined: Runtime vs Initial Neg-Jdet Pixels

Overlay both sweeps to see if runtime correlates with the number of
folding pixels regardless of how the folding was produced.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.scatter(m_negs, m_times, marker="o", s=80, label="Magnitude sweep", zorder=3)
ax.scatter(d_negs, d_times, marker="s", s=80, label="Density sweep", zorder=3)

ax.set_xlabel("Initial neg-Jdet pixels")
ax.set_ylabel("Time (s)")
ax.set_title("Correction Time vs Folding Severity")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

Key takeaways:
- **Magnitude**: higher displacement magnitudes create more folding and require more correction time and L2 deviation
- **Density**: more crossing regions increase both the number of folding pixels and total correction cost
- **Convergence**: at extreme severity levels the optimizer may not fully converge within default iteration limits